# 03. 파이썬 기초 - 예외처리와 파일 다루기

웹 요청은 **언제든 실패**할 수 있고, 수집한 데이터는 **파일로 저장**해야 합니다.
이 편에서 `try/except`, 파일 입출력, JSON, 환경변수를 익힙니다.

**다루는 내용**
1. 예외처리 try / except / finally
2. 파일 읽기/쓰기 (with open)
3. JSON 다루기 (json 모듈)
4. 경로와 폴더 (os 모듈)
5. 환경변수와 .env (보안)

## 1. 예외처리 try / except

오류가 날 수 있는 코드를 `try` 에 넣고, 오류 발생 시 `except` 로 처리합니다.
프로그램이 멈추지 않게 하는 안전장치입니다.

In [ ]:
# 오류가 나면 프로그램이 멈춘다 → try/except 로 감싸 안전하게 처리
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:          # 0으로 나누는 특정 오류만 처리
        print('0으로 나눌 수 없습니다.')
        result = None
    return result

print(safe_divide(10, 2))
print(safe_divide(10, 0))

In [ ]:
# 여러 예외 처리 + finally (성공/실패와 무관하게 항상 실행)
# 실제 스크래핑 코드의 requests 오류 처리와 같은 패턴입니다.
def process(value):
    try:
        number = int(value)          # 숫자로 변환 시도
        print('변환 성공:', number)
    except ValueError as e:          # as e : 오류 객체를 e 에 담아 내용 확인
        print('변환 실패:', e)
    finally:
        print('처리 종료\n')        # 항상 실행 (Selenium 의 driver.quit() 위치)

process('123')
process('abc')

## 2. 파일 읽기/쓰기 — with open

`with open(...)` 을 쓰면 파일을 자동으로 닫아줘 안전합니다.
한글이 깨지지 않도록 `encoding='utf-8'` 을 지정합니다.

In [ ]:
# 파일 쓰기 ('w' = write, 기존 내용 덮어씀)
with open('sample.txt', 'w', encoding='utf-8') as f:
    f.write('첫번째 줄\n')
    f.write('두번째 줄\n')
print('파일 저장 완료')

# 파일 읽기 ('r' = read)
with open('sample.txt', 'r', encoding='utf-8') as f:
    content = f.read()
print(content)

## 3. JSON 다루기

API 응답과 `data/` 폴더의 `.json` 파일은 JSON 형식입니다.
- `json.dump` : 파이썬 객체 → JSON 파일로 저장
- `json.load` : JSON 파일 → 파이썬 객체로 읽기

In [ ]:
import json

# 저장할 데이터 (딕셔너리들의 리스트 — 01편에서 배운 구조)
songs = [
    {'rank': 1, 'title': 'Dynamite', 'artist': 'BTS'},
    {'rank': 2, 'title': 'How You Like That', 'artist': 'BLACKPINK'},
]

# JSON 파일로 저장
# ensure_ascii=False : 한글을 그대로 저장 (True면 \uXXXX 로 깨져 보임)
# indent=2 : 사람이 읽기 좋게 들여쓰기
with open('songs_sample.json', 'w', encoding='utf-8') as f:
    json.dump(songs, f, ensure_ascii=False, indent=2)
print('JSON 저장 완료')

In [ ]:
# JSON 파일 읽기 → 다시 파이썬 리스트/딕셔너리로
with open('songs_sample.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(type(loaded))          # list
print(loaded[0]['title'])    # 첫 곡 제목
for s in loaded:
    print(s['rank'], s['title'], '-', s['artist'])

## 4. 경로와 폴더 — os 모듈

파일을 저장하기 전에 폴더가 있는지 확인하고 없으면 만듭니다.
스크래핑 코드의 `os.makedirs('data', exist_ok=True)` 가 그 예입니다.

In [ ]:
import os

# 폴더가 없으면 만들기 (exist_ok=True : 이미 있어도 오류 안 남)
os.makedirs('output_sample', exist_ok=True)
print('폴더 준비 완료')

# 경로 합치기 (운영체제에 맞게 / 또는 \ 처리)
path = os.path.join('output_sample', 'result.txt')
print('경로:', path)

# 파일/폴더 존재 확인
print('sample.txt 존재?', os.path.exists('sample.txt'))

## 5. 환경변수와 .env — 비밀정보 다루기 

API 키·비밀번호를 코드에 직접 쓰면 위험합니다.
`.env` 파일에 넣고 `os.getenv()` 로 불러오면 코드와 비밀정보를 분리할 수 있습니다.
(이 프로젝트의 `streamlit_book_search.py` 가 이 방식을 사용합니다.)

In [ ]:
# .env 파일 예시 (실제로는 아래 내용을 프로젝트 루트의 .env 에 저장)
#   CLIENT_ID=여기에_아이디
#   CLIENT_SECRET=여기에_비밀키

# python-dotenv 라이브러리로 .env 를 읽어 환경변수로 등록
from dotenv import load_dotenv
import os

load_dotenv()   # .env 파일을 읽어옴

# os.getenv('키') 로 값을 꺼낸다 (없으면 None)
client_id = os.getenv('CLIENT_ID')

# 비밀정보는 전체를 출력하지 말고 일부만 확인
if client_id:
    print('CLIENT_ID 앞 4자리:', client_id[:4])
else:
    print('CLIENT_ID 가 설정되어 있지 않습니다. (.env 확인)')

# ⚠️ 주의: .env 파일은 .gitignore 에 넣어 Git 에 올리지 않아야 합니다.

## 정리
- **try/except/finally**: 오류에 안전한 코드 (requests 실패 대비)
- **with open(..., encoding='utf-8')**: 파일 자동 닫힘 + 한글 처리
- **json.dump / json.load**: 파이썬 객체 ↔ JSON 파일
- **os.makedirs / path.join / path.exists**: 폴더·경로 안전 처리
- **dotenv + os.getenv**: 비밀정보를 코드와 분리

다음: `04python_basic_pandas기초.ipynb` (표 데이터 분석)